# Test III: Variational Quantum Classifier (VQC) with CUDA-Q

**Task:** Build a quantum machine learning model for 3-class gravitational lensing classification.

**Pipeline:**
```
Image (1, 150, 150) → Flatten → PCA (16 dims) → Angle Encoding (16 qubits)
    → Variational Circuit (data re-uploading, 3 layers) → Measure ⟨Z⟩ on 3 qubits → 3 classes
```

**Key design choices:**
- **Dimensionality reduction:** PCA reduces 22,500 pixels to 16 features (1 per qubit)
- **Data encoding:** Angle encoding with data re-uploading for higher expressivity
- **Circuit:** 16 qubits, 3 variational layers, ring CNOT entanglement
- **Readout:** ⟨Z⟩ on qubits 0, 1, 2 as logits for 3 classes
- **Gradient:** SPSA (2 circuit calls per backward, independent of param count)
- **Framework:** NVIDIA CUDA-Q (GPU-accelerated state vector simulation)

In [ ]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import cudaq
from cudaq import spin
import warnings
warnings.filterwarnings('ignore')

try:
    cudaq.set_target('nvidia')
    print('CUDA-Q target: nvidia (GPU)')
except:
    print('CUDA-Q target: default (CPU)')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch device: {device}')

## 1. Load Data & PCA

In [ ]:
CLASS_NAMES = ['no', 'sphere', 'vort']
DATA_DIR = 'data'
N_QUBITS = 16

def load_split(split):
    images, labels = [], []
    for ci, cname in enumerate(CLASS_NAMES):
        cdir = os.path.join(DATA_DIR, split, cname)
        for f in sorted(os.listdir(cdir)):
            if f.endswith('.npy'):
                img = np.load(os.path.join(cdir, f)).astype(np.float32).flatten()
                images.append(img)
                labels.append(ci)
    return np.array(images), np.array(labels)

print('Loading data...')
X_train_raw, y_train = load_split('train')
X_val_raw, y_val = load_split('val')
print(f'Train: {X_train_raw.shape}, Val: {X_val_raw.shape}')

In [ ]:
# PCA: 22500 dims → 16 dims
print(f'Applying PCA: {X_train_raw.shape[1]} → {N_QUBITS} dimensions')
pca = PCA(n_components=N_QUBITS)
X_train_pca = pca.fit_transform(X_train_raw)
X_val_pca = pca.transform(X_val_raw)

explained = pca.explained_variance_ratio_.sum()
print(f'Explained variance: {explained:.4f} ({explained*100:.1f}%)')

# Normalize PCA features to [0, 1] for angle encoding
pca_min = X_train_pca.min(axis=0)
pca_max = X_train_pca.max(axis=0)
X_train_norm = (X_train_pca - pca_min) / (pca_max - pca_min + 1e-8)
X_val_norm = (X_val_pca - pca_min) / (pca_max - pca_min + 1e-8)
X_val_norm = np.clip(X_val_norm, 0, 1)

print(f'Train PCA features: {X_train_norm.shape}, range [{X_train_norm.min():.3f}, {X_train_norm.max():.3f}]')
print(f'Val PCA features: {X_val_norm.shape}')

In [ ]:
# Visualize PCA explained variance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(range(1, N_QUBITS+1), pca.explained_variance_ratio_)
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Explained Variance Ratio')
ax1.set_title('PCA Components')

cumvar = np.cumsum(pca.explained_variance_ratio_)
ax2.plot(range(1, N_QUBITS+1), cumvar, 'bo-')
ax2.set_xlabel('Number of Components')
ax2.set_ylabel('Cumulative Explained Variance')
ax2.set_title(f'Cumulative: {cumvar[-1]*100:.1f}% with {N_QUBITS} components')
ax2.axhline(y=0.9, color='r', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 2. Quantum Circuit — VQC

**Architecture:**
- 16 qubits, each encoding one PCA feature via Ry rotation
- 3 variational layers with data re-uploading:
  - `Ry(feature_i × π + θ)` — data + trainable rotation
  - Ring CNOT entanglement: q0→q1→...→q15→q0
  - `Rz(θ)` — trainable rotation
- Readout: ⟨Z₀⟩, ⟨Z₁⟩, ⟨Z₂⟩ as logits for 3 classes
- Total: 96 trainable quantum parameters

In [ ]:
N_LAYERS = 3
N_CLASSES = 3
N_PARAMS = N_LAYERS * 2 * N_QUBITS  # 96
print(f'VQC: {N_QUBITS} qubits, {N_LAYERS} layers, {N_PARAMS} params')

@cudaq.kernel
def vqc_kernel(data: list[float], params: list[float],
               n_qubits: int, n_layers: int):
    """Variational Quantum Classifier kernel."""
    q = cudaq.qvector(n_qubits)
    for layer in range(n_layers):
        base = layer * 2 * n_qubits
        for i in range(n_qubits):
            ry(data[i] * 3.14159265 + params[base + i], q[i])
        for i in range(n_qubits - 1):
            x.ctrl(q[i], q[i + 1])
        x.ctrl(q[n_qubits - 1], q[0])
        for i in range(n_qubits):
            rz(params[base + n_qubits + i], q[i])

# Observables: Z on first 3 qubits → 3 class logits
CLASS_OBS = [spin.z(i) for i in range(N_CLASSES)]
COMBINED_OBS = sum(CLASS_OBS)

def run_vqc(data_list, params_list):
    """Run VQC on one sample. Returns 3 expectation values."""
    result = cudaq.observe(vqc_kernel, COMBINED_OBS,
                           data_list, params_list, N_QUBITS, N_LAYERS)
    return [result.expectation(obs) for obs in CLASS_OBS]

# Quick test
test_data = np.random.rand(N_QUBITS).tolist()
test_params = (np.random.randn(N_PARAMS) * 0.1).tolist()
out = run_vqc(test_data, test_params)
print(f'Test output (3 logits): {[f"{v:.4f}" for v in out]}')

## 3. SPSA Gradient & Training Loop

**SPSA (Simultaneous Perturbation Stochastic Approximation):**
- Perturb ALL parameters simultaneously with random ±ε
- Only 2 circuit evaluations per sample per backward (vs 2×N_params for parameter-shift)
- Gradient estimate: `g_k ≈ [f(θ+εΔ) - f(θ-εΔ)] / (2ε·Δ_k)`

In [ ]:
# Subsample for training (full val for evaluation)
TRAIN_SUBSET = 3000  # 1000 per class
BATCH_SIZE = 16

indices = []
for c in range(3):
    ci = np.where(y_train == c)[0]
    chosen = np.random.choice(ci, TRAIN_SUBSET // 3, replace=False)
    indices.extend(chosen.tolist())
np.random.shuffle(indices)

X_tr = torch.tensor(X_train_norm[indices], dtype=torch.float32)
y_tr = torch.tensor(y_train[indices], dtype=torch.long)
X_vl = torch.tensor(X_val_norm, dtype=torch.float32)
y_vl = torch.tensor(y_val, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_vl, y_vl), batch_size=BATCH_SIZE, shuffle=False)

print(f'Train: {len(X_tr)} samples, Val: {len(X_vl)} samples')
print(f'Batches per epoch: {len(train_loader)}')

In [ ]:
def vqc_forward_batch(X_batch, params_list):
    """Forward pass: run VQC on a batch. Returns (B, 3) logits."""
    B = X_batch.shape[0]
    logits = np.zeros((B, N_CLASSES))
    for b in range(B):
        exp_vals = run_vqc(X_batch[b].tolist(), params_list)
        for c in range(N_CLASSES):
            logits[b, c] = exp_vals[c]
    return logits


def spsa_gradient(X_batch, y_batch, params_np, epsilon=0.1):
    """Compute SPSA gradient estimate for quantum params.
    
    Also returns the loss at current params for logging.
    """
    B = X_batch.shape[0]
    X_np = X_batch.numpy()
    y_np = y_batch.numpy()
    
    # Random perturbation
    delta = np.random.choice([-1.0, 1.0], size=params_np.shape)
    params_plus = (params_np + epsilon * delta).tolist()
    params_minus = (params_np - epsilon * delta).tolist()
    
    # Forward at θ+εΔ and θ-εΔ
    logits_plus = vqc_forward_batch(X_np, params_plus)
    logits_minus = vqc_forward_batch(X_np, params_minus)
    
    # Cross-entropy loss for both
    def cross_entropy(logits, labels):
        # Softmax + NLL
        exp_l = np.exp(logits - logits.max(axis=1, keepdims=True))
        probs = exp_l / exp_l.sum(axis=1, keepdims=True)
        return -np.mean(np.log(probs[np.arange(len(labels)), labels] + 1e-10))
    
    loss_plus = cross_entropy(logits_plus, y_np)
    loss_minus = cross_entropy(logits_minus, y_np)
    
    # SPSA gradient: g_k = (L+ - L-) / (2ε * Δ_k)
    grad = (loss_plus - loss_minus) / (2.0 * epsilon) / delta
    
    # Estimate current loss as average
    loss_est = (loss_plus + loss_minus) / 2.0
    
    return grad, loss_est


def evaluate(params_np, loader):
    """Evaluate accuracy on a DataLoader."""
    correct, total = 0, 0
    all_probs, all_labels = [], []
    params_list = params_np.tolist()
    
    for X_batch, y_batch in loader:
        logits = vqc_forward_batch(X_batch.numpy(), params_list)
        # Softmax
        exp_l = np.exp(logits - logits.max(axis=1, keepdims=True))
        probs = exp_l / exp_l.sum(axis=1, keepdims=True)
        preds = probs.argmax(axis=1)
        correct += (preds == y_batch.numpy()).sum()
        total += len(y_batch)
        all_probs.append(probs)
        all_labels.append(y_batch.numpy())
    
    acc = correct / total
    return acc, np.concatenate(all_probs), np.concatenate(all_labels)

In [ ]:
# Benchmark: time one training step
test_X = X_tr[:BATCH_SIZE].numpy()
test_y = y_tr[:BATCH_SIZE].numpy()
test_params = np.random.randn(N_PARAMS) * 0.1

t0 = time.time()
logits = vqc_forward_batch(test_X, test_params.tolist())
fwd_time = time.time() - t0

t0 = time.time()
grad, loss = spsa_gradient(X_tr[:BATCH_SIZE], y_tr[:BATCH_SIZE], test_params)
step_time = time.time() - t0

steps_per_epoch = len(train_loader)
print(f'Forward (bs={BATCH_SIZE}): {fwd_time:.3f}s')
print(f'SPSA step (bs={BATCH_SIZE}): {step_time:.3f}s')
print(f'Steps/epoch: {steps_per_epoch}')
print(f'Estimated time/epoch: {steps_per_epoch * step_time:.0f}s = {steps_per_epoch * step_time / 60:.1f}min')

## 4. Training

In [ ]:
# SPSA optimizer with momentum
NUM_EPOCHS = 30
LR = 0.05
MOMENTUM = 0.9
EPSILON = 0.1

# Initialize
params = np.random.randn(N_PARAMS) * 0.1
velocity = np.zeros_like(params)
best_val_acc = 0.0
best_params = params.copy()

history = {'train_loss': [], 'val_acc': []}

# Resume from checkpoint if exists
ckpt_path = 'vqc_ckpt.npz'
start_epoch = 0
if os.path.exists(ckpt_path):
    ckpt = np.load(ckpt_path, allow_pickle=True)
    params = ckpt['params']
    velocity = ckpt['velocity']
    best_params = ckpt['best_params']
    best_val_acc = float(ckpt['best_val_acc'])
    start_epoch = int(ckpt['epoch'])
    history = ckpt['history'].item()
    print(f'Resumed from epoch {start_epoch}, best_val_acc={best_val_acc:.4f}')

print(f'Training VQC: {NUM_EPOCHS} epochs, lr={LR}, momentum={MOMENTUM}')
print(f'Params: {N_PARAMS}, Qubits: {N_QUBITS}, Layers: {N_LAYERS}')

In [ ]:
for epoch in range(start_epoch, NUM_EPOCHS):
    t0 = time.time()
    epoch_loss = 0.0
    n_steps = 0
    
    for X_batch, y_batch in train_loader:
        grad, loss = spsa_gradient(X_batch, y_batch, params, epsilon=EPSILON)
        
        # SGD with momentum
        velocity = MOMENTUM * velocity - LR * grad
        params = params + velocity
        
        epoch_loss += loss
        n_steps += 1
    
    avg_loss = epoch_loss / n_steps
    
    # Evaluate on val (subsample for speed during training)
    val_sub_loader = DataLoader(TensorDataset(X_vl[:1500], y_vl[:1500]),
                                batch_size=BATCH_SIZE, shuffle=False)
    val_acc, _, _ = evaluate(params, val_sub_loader)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_params = params.copy()
    
    history['train_loss'].append(avg_loss)
    history['val_acc'].append(val_acc)
    
    # Save checkpoint
    np.savez(ckpt_path,
             params=params, velocity=velocity,
             best_params=best_params, best_val_acc=best_val_acc,
             epoch=epoch + 1, history=history)
    
    elapsed = time.time() - t0
    print(f'Epoch {epoch+1:02d}/{NUM_EPOCHS} ({elapsed:.0f}s) | '
          f'Train Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f} | '
          f'Best: {best_val_acc:.4f}')

print(f'\nTraining complete. Best Val Accuracy: {best_val_acc:.4f}')

## 5. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], 'b-')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss (SPSA)')
ax1.grid(alpha=0.3)

ax2.plot(history['val_acc'], 'r-')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title(f'Validation Accuracy (Best: {best_val_acc:.4f})')
ax2.axhline(y=1/3, color='gray', linestyle='--', alpha=0.5, label='Random')
ax2.legend()
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Final Evaluation — ROC Curve & AUC

In [ ]:
# Evaluate best model on full val set
print('Evaluating best model on full validation set...')
full_val_loader = DataLoader(TensorDataset(X_vl, y_vl),
                              batch_size=BATCH_SIZE, shuffle=False)
final_acc, all_probs, all_labels = evaluate(best_params, full_val_loader)
all_labels_bin = label_binarize(all_labels, classes=[0, 1, 2])

print(f'Final Val Accuracy: {final_acc:.4f}')

In [ ]:
# ROC curves (one-vs-rest)
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#e41a1c', '#377eb8', '#4daf4a']

for i, class_name in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(all_labels_bin[:, i], all_probs[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=colors[i], lw=2,
            label=f'{class_name} (AUC = {roc_auc:.4f})')

macro_auc = roc_auc_score(all_labels_bin, all_probs,
                           multi_class='ovr', average='macro')
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curves — VQC with CUDA-Q (Macro AUC = {macro_auc:.4f})')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_vqc.png', dpi=150)
plt.show()

print(f'\nPer-class AUC:')
for i, class_name in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(all_labels_bin[:, i], all_probs[:, i])
    print(f'  {class_name}: {auc(fpr, tpr):.4f}')
print(f'Macro AUC: {macro_auc:.4f}')

## 7. Discussion

### Strategy & Design Choices

**Dimensionality Reduction — PCA:**
- Reduces 22,500 pixels to 16 features, one per qubit
- PCA is linear and fast; preserves the most variance in the fewest dimensions
- Features normalized to [0, 1] for angle encoding: θ = feature × π

**Data Encoding — Angle Encoding with Data Re-uploading:**
- Each PCA feature encoded as Ry(feature × π + θ_trainable) on its qubit
- Data re-uploading (re-encoding at each variational layer) provably increases the expressivity of the quantum model beyond a single encoding pass (Pérez-Salinas et al., 2020)

**Circuit Architecture — Variational Ansatz:**
- 16 qubits, 3 layers, each with Ry + ring CNOT + Rz
- Ring CNOT topology creates long-range entanglement between all qubits
- 96 trainable parameters total

**Readout — Pauli-Z Expectation Values:**
- ⟨Z₀⟩, ⟨Z₁⟩, ⟨Z₂⟩ serve as logits for 3 classes
- Softmax converts to probabilities for cross-entropy loss
- Using cudaq.observe() with combined observable for efficiency

**Gradient — SPSA:**
- Only 2 circuit evaluations per backward pass (vs 192 for parameter-shift with 96 params)
- Provides unbiased gradient estimate with much lower computational cost
- Trade-off: higher variance gradients, but compensated by momentum

**Why CUDA-Q:**
- GPU-accelerated state vector simulation (nvidia target)
- Native cudaq.observe() with analytic expectation values (no shot noise)
- Significantly faster than CPU-based quantum simulators